### PydanticOutputParser

In [ ]:
#!pip --version

pip 25.0.1 from C:\pkh20260902\ex0916\.venv\Lib\site-packages\pip (python 3.12)



In [ ]:
#!pip install dotenv

In [7]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# !pip install -U langchain langchain_openai

In [8]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OpenAI 키:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("LANGSMITH 키:", os.getenv("LANGSMITH_API_KEY")[:8] + "...")
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

OpenAI 키: sk-proj-...
LANGSMITH 키: lsv2_pt_...
LangSmith 프로젝트: test0914


In [ ]:
#!pip install langchain_teddynote

In [9]:
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("test0914")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [10]:
from langchain_teddynote.messages import stream_response
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("test0914")

llm = ChatOpenAI(temperature=0.1, model="gpt-4o-mini")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [11]:
email_conversation = """From: 김철수 (chulsoo.kim@bikecoporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일저 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. ... 다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 귀사 사무실에서 만나 이야기를 나눌 수 있을까요?
...
김철수
상무이사
바이크코퍼레이션
"""

In [12]:
from itertools import chain
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "다음의 이메일 내용 중 중요한 내용을 추출해 주세요.\n\n{email_conversation}"
)

llm = ChatOpenAI(temperature=0.1, model="gpt-4o-mini")

chain = prompt | llm

answer = chain.stream({"email_conversation": email_conversation})

output = stream_response(answer, return_output=True)

중요한 내용:

- 발신자: 김철수 (바이크코퍼레이션 상무)
- 수신자: 이은채 (테디인터내셔널)
- 주제: "ZENESIS" 자전거 유통 협력 및 미팅 제안
- 미팅 제안 일시: 다음 주 화요일(1월 15일) 오전 10시
- 미팅 장소: 귀사 사무실

In [13]:
class EmailSummary(BaseModel):
    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메이을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")

In [14]:
parser = PydanticOutputParser(pydantic_object=EmailSummary)

In [15]:
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메이을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required": ["person", "email", "subject", "summary", "date"]}
```


In [16]:
prompt = PromptTemplate.from_template(
    """
You are a helpful assistant. Please answer the following questions in KOREAN.

QUESTION:
{question}

EMAIL CONVERSATION:
{email_conversation}

FORMAT:
{format}
"""
)

In [17]:
prompt = prompt.partial(format=parser.get_format_instructions())
prompt

PromptTemplate(input_variables=['email_conversation', 'question'], input_types={}, partial_variables={'format': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메이을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required

In [18]:
chain = prompt | llm

response = chain.stream(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용 중 주요 내용을 추출해 주세요.",
    }
)

output = stream_response(response, return_output=True)

```json
{
  "person": "김철수",
  "email": "chulsoo.kim@bikecoporation.me",
  "subject": "\"ZENESIS\" 자전거 유통 협력 및 미팅 일저 제안",
  "summary": "김철수 상무가 이은채 대리님에게 다음 주 화요일(1월 15일) 오전 10시에 귀사 사무실에서 미팅을 제안함.",
  "date": "2024-01-15T10:00:00"
}
```

In [19]:
structured_output = parser.parse(output)
print(structured_output)

person='김철수' email='chulsoo.kim@bikecoporation.me' subject='"ZENESIS" 자전거 유통 협력 및 미팅 일저 제안' summary='김철수 상무가 이은채 대리님에게 다음 주 화요일(1월 15일) 오전 10시에 귀사 사무실에서 미팅을 제안함.' date='2024-01-15T10:00:00'


In [20]:
structured_output.person

'김철수'

In [21]:
structured_output.date

'2024-01-15T10:00:00'

In [22]:
structured_output.email

'chulsoo.kim@bikecoporation.me'

In [23]:
chain = prompt | llm | parser

response = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용 중 주요 내용을 추출해 주세요.",
    }
)

response

EmailSummary(person='김철수', email='chulsoo.kim@bikecoporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일저 제안', summary='김철수 상무가 이은채 대리님에게 다음 주 화요일(1월 15일) 오전 10시에 귀사 사무실에서 미팅을 제안함.', date='2024-01-15T10:00:00')

### with_structured_output() 바인딩

In [3]:
from langchain_openai import ChatOpenAI

In [24]:
llm = ChatOpenAI(
    temperature=0.1, model="gpt-4o-mini"
)

llm.invoke("What is the capital city of Norway?")

AIMessage(content='The capital city of Norway is Oslo.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 15, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_abc11bd8ce', 'id': 'chatcmpl-EOZMnIBWhOLISkwKs6xRIiMP3TdS2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0a7f9-5218-7af2-8f2b-2339c9282d25-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 8, 'total_tokens': 23, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [25]:
llm_with_structured = ChatOpenAI(
    temperature=0.1, model="gpt-4o-mini"
).with_structured_output(EmailSummary)

In [26]:
answer = llm_with_structured.invoke(email_conversation)
answer.person

'이은채'

### 쉼표로 구분된 리스트 출력 파서

In [1]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("test0914")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [2]:
output_parser = CommaSeparatedListOutputParser()

format_instructions = output_parser.get_format_instructions()

In [3]:
print(format_instructions)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [4]:
prompt = PromptTemplate(
    template="List five {subject}.\n{format_instructions}",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions},
)

In [5]:
model = ChatOpenAI(temperature=0.1)

chain = prompt | model | output_parser

In [6]:
answer = chain.invoke({"subject": "tourist sites in Norway"})
answer

['Geirangerfjord', 'Oslo', 'Bergen', 'Tromsø', 'Lofoten Islands']

In [7]:
for s in chain.stream({"subject": "tourist sites in Norway"}):
    print(s)

['Geirangerfjord']
['Oslo']
['Bergen']
['Tromsø']
['Lofoten Islands']


### 구조화된 출력 파서

In [ ]:
# !pip install langchain

In [ ]:
# !pip install langchain_classic

In [4]:
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("test0914")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [5]:
response_schemas = [
    ResponseSchema(name="answer", description="사용자의 질문에 대한 답변"),
    ResponseSchema(
        name="source",
        description="사용자의 질문에 답하기 위해 사용된 '출처', '웹사이트 주소'이어야 합니다.",
    ),
]

In [6]:
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

In [7]:
print(output_parser.get_format_instructions())

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"answer": string  // 사용자의 질문에 대한 답변
	"source": string  // 사용자의 질문에 답하기 위해 사용된 '출처', '웹사이트 주소'이어야 합니다.
}
```


In [8]:
format_instructions = output_parser.get_format_instructions()
prompt = PromptTemplate(
    template="answer the users question as best as possible.\n{format_instructions}\n{question}",
    input_variables=["question"],
    partial_variables={"format_instructions": format_instructions},
)

model = ChatOpenAI(temperature=0.1)
chain = prompt | model | output_parser

chain.invoke({"question": "What is the capital city of Norway?"})

{'answer': 'Oslo', 'source': 'https://en.wikipedia.org/wiki/Oslo'}

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("test0914")

model = ChatOpenAI(temperature=0.1, model="gpt-4o-mini")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [10]:
class Topic(BaseModel):
    description: str = Field(description="주제에 대한 간결한 설명")
    hashtags: str = Field(description="해시태그 형식의 키워드(2개 이상)")

In [11]:
question = "Please explain to me about how serious global warming is."

parser = JsonOutputParser(pydantic_object=Topic)
print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [13]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Please make the answer short."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | model | parser

answer = chain.invoke({"question": question})

In [14]:
answer["description"]

'Global warming is a critical issue that leads to climate change, affecting weather patterns, sea levels, and ecosystems. It poses serious risks to human health, food security, and biodiversity.'

### Pandas 데이터프레임 출력 Parser

In [15]:
import pprint
from typing import Any, Dict

import pandas as pd
from langchain_classic.output_parsers import PandasDataFrameOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("test0914")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [16]:
model = ChatOpenAI(temperature=0.1, model="gpt-3.5-turbo")

In [17]:
def format_parser_output(parser_output: Dict[str, Any]) -> None:
    for key in parser_output.keys():
        parser_output[key] = parser_output[key].to_dict()
    return pprint.PrettyPrinter(width=4, compact=True).pprint(parser_output)

In [19]:
df = pd.read_csv("./langchain-kr/03-OutputParser/data/titanic.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [20]:
parser = PandasDataFrameOutputParser(dataframe=df)

print(parser.get_format_instructions())

The output should be formatted as a string as the operation, followed by a colon, followed by the column or row to be queried on, followed by optional array parameters.
1. The column names are limited to the possible columns below.
2. Arrays must either be a comma-separated list of numbers formatted as [1,3,5], or it must be in range of numbers formatted as [0..4].
3. Remember that arrays are optional and not necessarily required.
4. If the column is not in the possible columns or the operation is not a valid Pandas DataFrame operation, return why it is invalid as a sentence starting with either "Invalid column" or "Invalid operation".

As an example, for the formats:
1. String "column:num_legs" is a well-formatted instance which gets the column num_legs, where num_legs is a possible column.
2. String "row:1" is a well-formatted instance which gets row 1.
3. String "column:num_legs[1,2]" is a well-formatted instance which gets the column num_legs for rows 1 and 2, where num_legs is a p

In [21]:
df_query = "Look up the Age column, please."

prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{question}\n",
    input_variables=["question"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    },
)

chain = prompt | model | parser

parser_output = chain.invoke({"question": df_query})

format_parser_output(parser_output)

{'Age': {0: 22.0,
         1: 38.0,
         2: 26.0,
         3: 35.0,
         4: 35.0,
         5: nan,
         6: 54.0,
         7: 2.0,
         8: 27.0,
         9: 14.0,
         10: 4.0,
         11: 58.0,
         12: 20.0,
         13: 39.0,
         14: 14.0,
         15: 55.0,
         16: 2.0,
         17: nan,
         18: 31.0,
         19: nan}}


In [22]:
df_query = "Retrieve the first row."
parser_output = chain.invoke({"question": df_query})
format_parser_output(parser_output)

{'0': {'Age': 22.0,
       'Cabin': nan,
       'Embarked': 'S',
       'Fare': 7.25,
       'Name': 'Braund, '
               'Mr. '
               'Owen '
               'Harris',
       'Parch': 0,
       'PassengerId': 1,
       'Pclass': 3,
       'Sex': 'male',
       'SibSp': 1,
       'Survived': 0,
       'Ticket': 'A/5 '
                 '21171'}}


In [23]:
df["Age"].head().mean()

np.float64(31.2)

In [25]:
df_query = "Retrieve the average of the Ages from row 0 to 4."
parser_output = chain.invoke({"question": df_query})
print(parser_output)

{'mean': np.float64(31.2)}


In [26]:
df_query = "Calculate average 'Fare' rate."
parser_output = chain.invoke({"question": df_query})
print(parser_output)

{'mean': np.float64(22.19937)}


In [27]:
df["Fare"].mean()

np.float64(22.19937)

### 날짜 형식 출력 파서

In [28]:
from langchain_classic.output_parsers import DatetimeOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("test0914")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [29]:
output_parser = DatetimeOutputParser()
output_parser.format = "%Y-%m-%d"

In [30]:
print(output_parser.get_format_instructions())

Write a datetime string that matches the following pattern: '%Y-%m-%d'.

Examples: 2026-09-16, 2025-09-16, 2026-09-15

Return ONLY this string, no other words!


In [31]:
template = """Answer the users question:

#Format Instructions:
{format_instructions}

#Question:
{question}

#Answer:"""

prompt = PromptTemplate.from_template(
    template,
    partial_variables={
        "format_instructions": output_parser.get_format_instructions()
    },
)

prompt

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={'format_instructions': "Write a datetime string that matches the following pattern: '%Y-%m-%d'.\n\nExamples: 2026-09-16, 2025-09-16, 2026-09-15\n\nReturn ONLY this string, no other words!"}, template='Answer the users question:\n\n#Format Instructions:\n{format_instructions}\n\n#Question:\n{question}\n\n#Answer:')

In [32]:
chain = prompt | ChatOpenAI() | output_parser

output = chain.invoke({"question": "The year Google was founded"})

In [33]:
output.strftime("%Y-%m-%d")

'1998-09-04'

### 열거형 출력 파서

In [34]:
from enum import Enum
from langchain_classic.output_parsers.enum import EnumOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv
logging.langsmith("test0914")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [35]:
class Colors(Enum):
    RED = "빨간색"
    GREEN = "초록색"
    BLUE = "파란색"

In [36]:
Colors.RED

<Colors.RED: '빨간색'>

In [37]:
parser = EnumOutputParser(enum=Colors)
parser.get_format_instructions()

'Select one of the following options: 빨간색, 초록색, 파란색'

In [38]:
prompt = PromptTemplate.from_template(
    """다음의 물체는 어떤 색깔인가요?
    
Object: {object}

Instructions: {instructions}"""
).partial(instructions=parser.get_format_instructions())

chain = prompt | ChatOpenAI() | parser

In [39]:
response = chain.invoke({"object": "Sky"})
print(response)

Colors.BLUE


In [40]:
type(response)

<enum 'Colors'>

In [41]:
response.value

'파란색'